# 2026-10-14_2 scTenifoldKnk MG 虚拟敲除结果整理与绘图

基于已有 scTenifoldKnk MEF2C pilot 输出（central / peripheral 两组独立 GRN），完成统一统计与绘图，不重新建网、不重新 KO、不修改任何上游结果。

- Plot 1：各 TF KO 后显著受扰动基因数量（p.adj < 0.05，排除 TF 自身）
- Plot 2：上游预测 target DEG 的扰动百分位（依赖 TF_TARGET_FILE，当前未提供则跳过）
- Plot 3A/3B：显著扰动基因与上游区域 DEG 的重叠（DEG share / DEG coverage）

上游输入：
- DEG_FILE：MG 区域 DEG 的 subclass × direction 分类（DM / DAL / AAM × Central-high / Peripheral-high）
- TF_TARGET_FILE：TF–target 关系表（尚未提供，格式要求见 Plot 2 cell）

注意：结果表述为 virtual KO 的预测网络扰动，不是真实 KO 实验验证；central-high / peripheral-high 指上游 DEG 方向，不是 KO 后的预测表达方向。

In [ ]:
# ---- 1. 配置：路径 / 阈值 / 颜色（集中修改处） --------------------------------
suppressPackageStartupMessages({
  library(data.table)
  library(ggplot2)
})
set.seed(1)

## 输入
KNK_DIR  <- "/project/work/2026-10-13_scTenifoldKnk/Data/"   # scTenifoldKnk pilot 输出目录
DEG_FILE <- "/project/work/2026-07-29_trajectory_class/Data/2026-07-30_1_3subclass_target_gene.csv"
TF_TARGET_FILE <- NA   # 上游 TF-target 关系表尚未提供；提供后改为文件路径
TARGET_CELLTYPE <- "MG"

## 输出
OUT_DIR  <- "/project/work/2026-10-13_scTenifoldKnk/Data/"
PLOT_DIR <- "/project/work/2026-10-13_scTenifoldKnk/Plot/"
DATE     <- "2026-10-14"

## 统计阈值
PADJ_CUTOFF <- 0.05   # 显著扰动主阈值（p.adj）；不设 FC / distance 硬阈值

## 上游分类映射（沿用项目口径：fovea + macula = central）
SUBCLASS_MAP <- c("Developmental maintained"      = "DM",
                  "Developmental aging-lost"      = "DAL",
                  "Adult acquired and maintained" = "AAM")
DIRECTION_MAP <- c("Macular direction"    = "Central-high",
                   "Peripheral direction" = "Peripheral-high")

## 颜色（沿用 /project/work/color.ipynb）
cols.region    <- c(central = "#6A3D9A", peripheral = "#E69F00")          # cols.tissue.merged
cols.subclass  <- c(DM = "#3C5488", DAL = "#E64B35", AAM = "#00A087")     # cols.subclass.short
cols.heat      <- c("#313695", "#74ADD1", "#FFFFBF", "#F46D43", "#A50026") # cols.heat.cor
region_labels  <- c(central = "Central KO", peripheral = "Peripheral KO")

## 绘图全局参数
BASE_SIZE <- 10

dir.create(OUT_DIR,  recursive = TRUE, showWarnings = FALSE)
dir.create(PLOT_DIR, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# ---- 2. 读取全部 TF 的 diffRegulation 结果并做结构检查 -------------------------
## 自动发现文件，兼容后续新增 TF（命名：<date>_MG_<region>_<TF>_diffRegulation.tsv）
knk_files <- list.files(KNK_DIR, pattern = "_MG_(central|peripheral)_.+_diffRegulation\\.tsv$",
                        full.names = TRUE)
if (length(knk_files) == 0) stop("未在 KNK_DIR 找到 diffRegulation 文件")

knk <- rbindlist(lapply(knk_files, function(f) {
  b <- basename(f)
  m <- regmatches(b, regexec("_MG_(central|peripheral)_(.+)_diffRegulation", b))[[1]]
  d <- fread(f)
  miss <- setdiff(c("gene", "distance", "FC", "p.adj"), names(d))
  if (length(miss) > 0) stop(b, " 缺少必需列: ", paste(miss, collapse = ", "))
  d[, .(TF = m[3], region = m[2], gene, distance, FC, p.adj)]
}))
knk[, region := factor(region, levels = c("central", "peripheral"))]

cat("发现文件数:", length(knk_files), "\n")
print(knk[, .(n_genes = .N), by = .(TF, region)])
if (anyDuplicated(knk[, .(TF, region, gene)]) > 0) stop("存在重复的 TF x region x gene 记录")
cat("p.adj 缺失:", sum(is.na(knk$p.adj)), "| FC 缺失:", sum(is.na(knk$FC)),
    "| FC 非有限:", sum(!is.finite(knk$FC)), "| p.adj 超出 [0,1]:", sum(knk$p.adj < 0 | knk$p.adj > 1, na.rm = TRUE), "\n")

## 排除被直接敲除的 TF 本身（不计入该 KO 的扰动统计）
knk <- knk[gene != TF]
cat("排除 KO 的 TF 自身后：\n")
print(knk[, .(n_tested = .N), by = .(TF, region)])

In [ ]:
# ---- 3. Plot 1: 各 TF KO 后显著受扰动基因数量 ----------------------------------
sig_summary <- knk[, .(
  n_tested      = .N,
  n_significant = sum(p.adj < PADJ_CUTOFF),
  pct_significant = round(100 * sum(p.adj < PADJ_CUTOFF) / .N, 2)
), by = .(TF, region)][order(TF, region)]

fwrite(sig_summary, file.path(OUT_DIR, paste0(DATE, "_01_KO_significant_gene_summary.tsv")), sep = "\t")
print(sig_summary)

## 按显著基因总数降序排列 TF
tf_order <- sig_summary[, .(tot = sum(n_significant)), by = TF][order(-tot), TF]

p1 <- ggplot(sig_summary[, .(TF = factor(TF, levels = tf_order), region, n_significant)],
             aes(TF, n_significant, fill = region)) +
  geom_col(position = position_dodge(0.75), width = 0.65, color = "grey20", linewidth = 0.2) +
  geom_text(aes(label = n_significant), position = position_dodge(0.75), vjust = -0.5, size = 3.4) +
  scale_fill_manual(values = cols.region, labels = region_labels, name = NULL) +
  scale_y_continuous(expand = expansion(mult = c(0.02, 0.15))) +
  labs(title = "Significantly perturbed genes per virtual knockout",
       subtitle = sprintf("MG | p.adj < %s | knocked-out TF excluded", PADJ_CUTOFF),
       x = NULL, y = "N significantly perturbed genes") +
  theme_bw(base_size = BASE_SIZE) +
  theme(legend.position = "top")
print(p1)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot1_KO_significant_gene_count.pdf")), p1,
       width = 4.6, height = 3.6)

In [ ]:
# ---- 4. 读取上游区域 DEG 分类并统计与 KO 检验基因的匹配 ------------------------
if (!file.exists(DEG_FILE)) stop("未找到 DEG_FILE: ", DEG_FILE)
deg <- fread(DEG_FILE, header = TRUE)[celltype == TARGET_CELLTYPE, .(gene, subclass, direction_class)]

## 映射为项目简称；未映射的明确报出
deg[, subclass_short := SUBCLASS_MAP[subclass]]
deg[, deg_direction  := DIRECTION_MAP[direction_class]]
cat("MG DEG 行数:", nrow(deg), "| 唯一基因数:", uniqueN(deg$gene), "\n")
cat("未映射 subclass:", sum(is.na(deg$subclass_short)),
    "| 未映射 direction:", sum(is.na(deg$deg_direction)), "\n")
if (anyDuplicated(deg$gene) > 0) {
  cat("一基因多行（跨 subclass 重复）:", sum(duplicated(deg$gene)), "行，分组统计按唯一基因去重\n")
}

## 版本 B（direction-only）冲突基因：同一基因在不同 subclass 中方向相反，从互斥分组中剔除
conflict_genes <- deg[, .(n_dir = uniqueN(deg_direction)), by = gene][n_dir > 1, gene]
cat("方向冲突基因数:", length(conflict_genes),
    if (length(conflict_genes) > 0) paste("->", paste(conflict_genes, collapse = ", ")) else "", "\n")

## 与每次 KO 检验基因集合的匹配（central / peripheral 为独立 gene universe，分别统计）
deg_uniq <- unique(deg$gene)
match_dt <- knk[, .(
  n_tested     = .N,
  n_DEG_tested = sum(deg_uniq %in% gene),
  n_DEG_sig    = sum(gene %in% deg_uniq & p.adj < PADJ_CUTOFF)
), by = .(TF, region)]
match_dt[, n_DEG_total := length(deg_uniq)]
match_dt[, match_rate_pct := round(100 * n_DEG_tested / n_DEG_total, 1)]
print(match_dt)

## 未进入任何 KO 检验的 DEG 基因
deg_untested <- setdiff(deg_uniq, unique(knk$gene))
cat("DEG 未进入任何 KO 检验的基因数:", length(deg_untested), "\n")

## 05: 输入 QC 与基因匹配长表
qc05 <- rbind(
  match_dt[, .(section = "gene_matching", TF, region,
               item = "n_tested / n_DEG_total / n_DEG_tested / n_DEG_significant / match_rate_pct",
               value = sprintf("%d / %d / %d / %d / %.1f", n_tested, n_DEG_total,
                               n_DEG_tested, n_DEG_sig, match_rate_pct))],
  data.table(
    section = "input_qc", TF = NA_character_, region = NA_character_,
    item = c("DEG_file", "n_rows_MG", "n_unique_genes", "n_duplicate_gene_rows",
             "n_conflict_direction_genes", "conflict_genes",
             "n_ENSG_style_ids", "DEG_genes_not_in_any_KO_tested",
             "TF_TARGET_file"),
    value = c(DEG_FILE, nrow(deg), length(deg_uniq), sum(duplicated(deg$gene)),
              length(conflict_genes), paste(conflict_genes, collapse = ","),
              sum(grepl("^ENSG", deg_uniq)), length(deg_untested),
              if (is.character(TF_TARGET_FILE)) TF_TARGET_FILE else "not provided (Plot 2 skipped)")
  )
)
fwrite(qc05, file.path(OUT_DIR, paste0(DATE, "_05_Input_QC_and_gene_matching.tsv")), sep = "\t")
print(qc05)

## Plot 2：上游 target DEG 的扰动强度（依赖 TF_TARGET_FILE）

上游 TF–target 关系表当前尚未提供。需要的最小格式：

| 列名 | 说明 |
|:--|:--|
| `TF` | TF 名称（须与 KO 输出文件名中的 TF 一致） |
| `target_gene` | 靶基因 symbol |
| `subclass`（可选） | 不提供时自动按基因从 DEG_FILE 匹配 |
| `direction`（可选） | 不提供时自动按基因从 DEG_FILE 匹配 |

提供 `TF_TARGET_FILE` 路径后重跑本 cell 即可生成 02 表与 Dotplot；当前缺失时不虚构数据、直接跳过。

In [ ]:
# ---- 5. Plot 2: 上游预测 target DEG 的扰动百分位（依赖 TF_TARGET_FILE）----------
## 扰动百分位 = 在该 TF 两个 region 共同检验的基因集合内按 FC 降序排名换算（top = 100%）
## 百分位只反映网络内相对排名，不能把两 region 差值解释为 KO 效应大小或区域间显著差异
if (!is.character(TF_TARGET_FILE) || !file.exists(TF_TARGET_FILE)) {
  cat("【跳过 Plot 2】TF_TARGET_FILE 未提供或不存在，不虚构上游数据。\n")
  cat("  需要格式：CSV/TSV，必需列 TF、target_gene；可选列 subclass、direction（见上方说明）。\n")
} else {
  tf_target <- fread(TF_TARGET_FILE)
  miss <- setdiff(c("TF", "target_gene"), names(tf_target))
  if (length(miss) > 0) stop("TF_TARGET_FILE 缺少必需列: ", paste(miss, collapse = ", "))
  tf_target <- unique(tf_target[, .(TF, target_gene)])

  ## 分类信息（target 不在 DEG_FILE 中则为 NA）
  cls <- unique(deg[!is.na(subclass_short), .(gene, subclass_short, deg_direction)])
  tf_target <- merge(tf_target, cls, by.x = "target_gene", by.y = "gene", all.x = TRUE)
  cat("TF_TARGET 中的 TF:", paste(unique(tf_target$TF), collapse = ", "), "\n")
  cat("无对应 KO 结果、被剔除的 TF:",
      paste(setdiff(unique(tf_target$TF), unique(knk$TF)), collapse = ", "), "\n")
  tf_target <- tf_target[TF %in% unique(knk$TF)]

  ## 每个 TF 的 central/peripheral 共同检验基因集合（百分位在同一基因集合上计算）
  common_by_tf <- sapply(unique(knk$TF), function(tf)
    intersect(knk[TF == tf & region == "central", gene],
              knk[TF == tf & region == "peripheral", gene]), simplify = FALSE)
  cat("各 TF 共同检验基因数:", paste(sapply(common_by_tf, length), collapse = ", "), "\n")

  ## 全量共同基因的排名百分位（rank 基于 FC 降序）
  pct_dt <- rbindlist(lapply(names(common_by_tf), function(tf) {
    rbindlist(lapply(c("central", "peripheral"), function(rg) {
      x <- knk[TF == tf & region == rg & gene %in% common_by_tf[[tf]]]
      x[, .(TF, region, gene, rank_FC = frank(-FC, ties.method = "first"), n_common = .N)]
    }))
  }))
  pct_dt[, percentile := round(100 * (n_common - rank_FC) / pmax(n_common - 1, 1), 1)]

  ## target 关联 KO 结果（仅保留在该 TF x region 中实际被检验的 target）
  target_long <- merge(tf_target, knk, by.x = c("TF", "target_gene"), by.y = c("TF", "gene"))
  cat("匹配到 KO 检验的 target 基因:", uniqueN(target_long$target_gene), "/",
      uniqueN(tf_target$target_gene), "\n")
  target_long <- merge(target_long, pct_dt,
                       by.x = c("TF", "region", "target_gene"), by.y = c("TF", "region", "gene"))
  target_long[, significant := p.adj < PADJ_CUTOFF]
  setcolorder(target_long, c("TF", "region", "target_gene", "subclass_short", "deg_direction",
                             "FC", "distance", "p.adj", "rank_FC", "n_common", "percentile", "significant"))
  fwrite(target_long, file.path(OUT_DIR, paste0(DATE, "_02_TF_target_perturbation_long.tsv")), sep = "\t")

  ## 每个 TF x region 显著受扰动的 target 列表
  cat("显著受扰动的 target DEG：\n")
  print(target_long[(significant)][order(TF, region, -percentile),
                                  .(TF, region, target_gene, deg_direction, FC, p.adj, percentile)])

  ## 可选筛选（按需修改）
  PLOT2_TF_FILTER        <- NULL   # 例: c("MEF2C")
  PLOT2_SUBCLASS_FILTER  <- NULL   # 例: c("DM", "AAM")
  PLOT2_DIRECTION_FILTER <- NULL   # 例: c("Central-high")
  p2dt <- copy(target_long)
  if (!is.null(PLOT2_TF_FILTER))        p2dt <- p2dt[TF %in% PLOT2_TF_FILTER]
  if (!is.null(PLOT2_SUBCLASS_FILTER))  p2dt <- p2dt[subclass_short %in% PLOT2_SUBCLASS_FILTER]
  if (!is.null(PLOT2_DIRECTION_FILTER)) p2dt <- p2dt[deg_direction %in% PLOT2_DIRECTION_FILTER]

  ## Dotplot：y 按两 region 平均百分位排序；颜色 = 上游 DEG 方向；大小 = 百分位；形状 = 显著性
  gene_ord <- p2dt[, .(m = mean(percentile)), by = target_gene][order(m), target_gene]
  p2 <- ggplot(p2dt[, .(TF, region, target_gene = factor(target_gene, levels = gene_ord),
                        deg_direction, percentile, significant)],
               aes(region, target_gene, color = deg_direction)) +
    geom_point(aes(size = percentile, shape = significant), stroke = 0.7) +
    scale_color_manual(values = c("Central-high" = unname(cols.region["central"]),
                                  "Peripheral-high" = unname(cols.region["peripheral"])),
                       name = "Upstream DEG direction", na.translate = FALSE) +
    scale_shape_manual(values = c("TRUE" = 16, "FALSE" = 1), labels = c("TRUE" = "yes", "FALSE" = "no"),
                       name = sprintf("p.adj < %s", PADJ_CUTOFF)) +
    scale_size_continuous(limits = c(0, 100), name = "Perturbation percentile (%)") +
    scale_x_discrete(labels = region_labels) +
    facet_wrap(~ TF, scales = "free_y") +
    labs(title = "Perturbation percentile of upstream predicted target DEGs",
         subtitle = "Percentile ranked by FC within genes tested in both KO regions",
         x = NULL, y = NULL) +
    theme_bw(base_size = BASE_SIZE) +
    theme(legend.position = "right", panel.grid.minor = element_blank())
  print(p2)
  ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot2_target_perturbation_percentile_dotplot.pdf")), p2,
         width = 6, height = 1.6 + 0.22 * length(gene_ord))
}

In [ ]:
# ---- 6. Plot 3 统计: 显著扰动基因与区域 DEG 的重叠（share / coverage）-----------
## S = 该 TF 该 region KO 后显著受扰动基因（p.adj < 阈值，已排除 TF 自身）
## D = 上游 DEG 分组；D_tested = D 中实际进入该次 KO 检验的基因
## DEG share    = |S ∩ D| / |S|        （显著扰动基因中 DEG 占比）
## DEG coverage = |S ∩ D| / |D_tested| （被显著扰动的 DEG 比例）
## 分母为 0 时输出 NA，不强制设为 0
ko_stats <- knk[, .(sig_list    = list(gene[p.adj < PADJ_CUTOFF]),
                    tested_list = list(gene)), by = .(TF, region)]
cat("各 TF x region 显著基因数:\n")
print(ko_stats[, .(TF, region, n_significant = lengths(sig_list))])

## 版本 A: subclass x direction（分组不互斥：同一基因可属于多个 subclass 组，比例不可相加）
groupsA <- unique(deg[!is.na(subclass_short) & !is.na(deg_direction),
                      .(subclass_short, deg_direction)])[order(subclass_short, deg_direction)]
resA <- list()
for (i in seq_len(nrow(ko_stats))) {
  S  <- ko_stats$sig_list[[i]]
  Tg <- ko_stats$tested_list[[i]]
  for (j in seq_len(nrow(groupsA))) {
    D <- deg[subclass_short == groupsA$subclass_short[j] &
             deg_direction == groupsA$deg_direction[j], unique(gene)]
    resA[[length(resA) + 1L]] <- data.table(
      ko_stats[i, .(TF, region)],
      subclass_short = groupsA$subclass_short[j], deg_direction = groupsA$deg_direction[j],
      n_significant = length(S), n_DEG = length(D),
      n_DEG_tested  = sum(D %in% Tg), n_overlap = sum(D %in% S),
      overlap_genes = paste(intersect(S, D), collapse = ","))
  }
}
resA <- rbindlist(resA)
resA[, deg_share_pct    := ifelse(n_significant > 0, round(100 * n_overlap / n_significant, 1), NA_real_)]
resA[, deg_coverage_pct := ifelse(n_DEG_tested  > 0, round(100 * n_overlap / n_DEG_tested, 1), NA_real_)]
fwrite(resA, file.path(OUT_DIR, paste0(DATE, "_03_DEG_overlap_subclass_direction.tsv")), sep = "\t")
print(resA)

## 版本 B: 仅 direction（各 subclass 合并取基因并集去重；冲突基因已剔除并单独导出）
degB <- deg[!gene %in% conflict_genes]
resB <- list()
for (i in seq_len(nrow(ko_stats))) {
  S  <- ko_stats$sig_list[[i]]
  Tg <- ko_stats$tested_list[[i]]
  for (g in c("Central-high", "Peripheral-high")) {
    D <- degB[deg_direction == g, unique(gene)]
    resB[[length(resB) + 1L]] <- data.table(
      ko_stats[i, .(TF, region)], deg_direction = g,
      n_significant = length(S), n_DEG = length(D),
      n_DEG_tested  = sum(D %in% Tg), n_overlap = sum(D %in% S),
      overlap_genes = paste(intersect(S, D), collapse = ","))
  }
}
resB <- rbindlist(resB)
resB[, deg_share_pct    := ifelse(n_significant > 0, round(100 * n_overlap / n_significant, 1), NA_real_)]
resB[, deg_coverage_pct := ifelse(n_DEG_tested  > 0, round(100 * n_overlap / n_DEG_tested, 1), NA_real_)]
fwrite(resB, file.path(OUT_DIR, paste0(DATE, "_04_DEG_overlap_direction_only.tsv")), sep = "\t")
cat("版本 B 剔除的冲突方向基因:", length(conflict_genes), "个（名单见 05 表）\n")
print(resB)

In [ ]:
# ---- 7. Plot 3A: Subclass x Direction 的 DEG share / coverage ------------------
sub_ord <- names(cols.subclass)
dir_ord <- c("Central-high", "Peripheral-high")
lvA <- paste0(rep(sub_ord, each = length(dir_ord)), " - ", rep(dir_ord, length(sub_ord)))

## 单行 group 标签 + 旋转 45 度，避免相邻 direction 文字重叠
plotA <- rbind(
  resA[, .(TF, region, deg_group = paste0(subclass_short, " - ", deg_direction),
           metric = "Share (%)",    value = deg_share_pct,    n_overlap)],
  resA[, .(TF, region, deg_group = paste0(subclass_short, " - ", deg_direction),
           metric = "Coverage (%)", value = deg_coverage_pct, n_overlap)]
)
plotA[, deg_group := factor(deg_group, levels = lvA)]

p3a <- ggplot(plotA, aes(deg_group, TF, fill = value)) +
  geom_tile(color = "white", linewidth = 0.6) +
  geom_text(aes(label = ifelse(is.na(value), "NA", sprintf("%.1f%%\n(n=%d)", value, n_overlap))),
            size = 2.6, lineheight = 0.85) +
  facet_grid(metric ~ region, labeller = labeller(region = region_labels)) +
  scale_fill_gradientn(colors = colorRampPalette(cols.heat)(100),
                       limits = c(0, 100), na.value = "grey85", name = "Percent (%)") +
  labs(title = "Overlap with regional DEGs, by subclass and direction",
       subtitle = "DEG share = overlap / |S|; DEG coverage = overlap / |D_tested|; groups non-exclusive; NA = zero denominator",
       x = "Upstream DEG group (subclass x direction)", y = NULL) +
  theme_bw(base_size = BASE_SIZE) +
  theme(axis.text.x = element_text(size = 7.5, angle = 45, hjust = 1, vjust = 1),
        strip.text = element_text(size = 8),
        strip.text.y = element_text(angle = 0),
        legend.position = "right")
print(p3a)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot3A_DEG_overlap_subclass_direction.pdf")), p3a,
       width = 7.8, height = 3.1 + 0.55 * nrow(unique(plotA[, .(TF)])))

In [ ]:
# ---- 8. Plot 3B: Direction-only 的 DEG share / coverage ------------------------
plotB <- rbind(
  resB[, .(TF, region, deg_group = deg_direction, metric = "Share (%)",
           value = deg_share_pct, n_overlap)],
  resB[, .(TF, region, deg_group = deg_direction, metric = "Coverage (%)",
           value = deg_coverage_pct, n_overlap)]
)
plotB[, deg_group := factor(deg_group, levels = dir_ord)]

p3b <- ggplot(plotB, aes(deg_group, TF, fill = value)) +
  geom_tile(color = "white", linewidth = 0.6) +
  geom_text(aes(label = ifelse(is.na(value), "NA", sprintf("%.1f%%\n(n=%d)", value, n_overlap))),
            size = 2.7, lineheight = 0.85) +
  facet_grid(metric ~ region, labeller = labeller(region = region_labels)) +
  scale_fill_gradientn(colors = colorRampPalette(cols.heat)(100),
                       limits = c(0, 100), na.value = "grey85", name = "Percent (%)") +
  labs(title = "Overlap with regional DEGs, by direction only",
       subtitle = "Subclasses merged (gene union, deduplicated); conflict-direction genes excluded; NA = zero denominator",
       x = "Upstream DEG direction", y = NULL) +
  theme_bw(base_size = BASE_SIZE) +
  theme(axis.text.x = element_text(size = 8.5, angle = 45, hjust = 1, vjust = 1),
        strip.text = element_text(size = 8),
        strip.text.y = element_text(angle = 0),
        legend.position = "right")
print(p3b)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot3B_DEG_overlap_direction_only.pdf")), p3b,
       width = 5.8, height = 3.1 + 0.55 * nrow(unique(plotB[, .(TF)])))

In [ ]:
# ---- 9. 输出文件汇总 ------------------------------------------------------------
cat("=== 本次输出（", DATE, "前缀）===\n")
cat("-- 统计表 --\n")
print(list.files(OUT_DIR, pattern = paste0("^", DATE, "_[0-9]{2}_"), full.names = TRUE))
cat("-- 图形（PDF）--\n")
print(list.files(PLOT_DIR, pattern = paste0("^", DATE, "_Plot"), full.names = TRUE))

cat("\n说明：\n")
cat("1. 以上结果为 scTenifoldKnk virtual KO 的预测网络扰动，不是真实 KO 实验验证。\n")
cat("2. central-high / peripheral-high 为上游 DEG 方向；KO 输出中的 direction 列为 KO 预测方向，本分析未使用。\n")
cat("3. central / peripheral 为独立 GRN，其 FC、distance、百分位不可直接相减解释为区域间 KO 效应差异。\n")
cat("4. Plot 2 依赖的 TF_TARGET_FILE 尚未提供，待上游提供后填入路径重跑即可。\n")

## 代表性 target DEG 扰动可视化（APOE / ABCB11）

展示 MEF2C virtual KO 后 APOE、ABCB11 在 central / peripheral 两个独立 GRN 中的相对网络扰动排名。

- 扰动百分位：每次 KO 内独立计算，背景为该次检验的全部 tested genes（沿用前文逻辑，已排除 KO 的 TF 本身），按 FC 降序换算，排名最靠前 = 100%。
- 百分位只反映网络内相对排名（network perturbation），不代表扰动绝对强度，也不表示 KO 后表达上调/下调。
- 本模块只在 Notebook 内展示，不生成新的数据表文件；图片输出 PDF + PNG。

In [ ]:
# ---- 10. 代表性 target DEG 的扰动百分位与完整参数 ------------------------------
REP_TF <- "MEF2C"                    # 本次展示的 KO TF
target_genes <- c("APOE", "ABCB11")  # 代表性 target DEG

stopifnot(all(c("gene", "distance", "FC", "p.adj") %in% names(knk)))

## 质量检查：基因缺失时 warning + NA，不静默删除、不视为未显著
for (g in target_genes) {
  for (rg in c("central", "peripheral")) {
    if (nrow(knk[TF == REP_TF & region == rg & gene == g]) == 0) {
      warning(sprintf("%s 不在 %s (%s KO) 的 diffRegulation 中，对应参数记为 NA", g, REP_TF, rg))
    }
  }
}

## 扰动百分位：每次 KO 内独立计算，全部 tested genes 为背景，按 FC 降序换算
knk_pct <- knk[TF == REP_TF][, {
  rk <- frank(-FC, ties.method = "first")
  .(gene, distance, FC, p.adj,
    Perturbation_percentile = round(100 * (.N - rk) / pmax(.N - 1, 1), 1))
}, by = region]

## Task 2: 完整参数表（仅 Notebook 打印，不保存文件）
rep_tab <- rbindlist(lapply(target_genes, function(g) {
  rbindlist(lapply(c("central", "peripheral"), function(rg) {
    x <- knk_pct[region == rg & gene == g]
    if (nrow(x) == 0) {
      data.table(Gene = g, Region = region_labels[[rg]], distance = NA_real_,
                 FC = NA_real_, p.adj = NA_real_,
                 Perturbation_percentile = NA_real_, Significant = NA)
    } else {
      data.table(Gene = g, Region = region_labels[[rg]], distance = x$distance,
                 FC = x$FC, p.adj = x$p.adj,
                 Perturbation_percentile = x$Perturbation_percentile,
                 Significant = x$p.adj < PADJ_CUTOFF)
    }
  }))
}))
rep_tab[, Gene := factor(Gene, levels = target_genes)]
rep_tab[, Region := factor(Region, levels = unname(region_labels))]
print(rep_tab)

In [ ]:
# ---- 11. Plot 4: 代表性 target DEG 扰动百分位柱状图 ----------------------------
p4 <- ggplot(rep_tab[!is.na(Perturbation_percentile)],
             aes(Gene, Perturbation_percentile, fill = Region)) +
  geom_col(position = position_dodge(0.75), width = 0.65, color = "grey20", linewidth = 0.2) +
  geom_text(aes(label = sprintf("%.1f", Perturbation_percentile)),
            position = position_dodge(0.75), vjust = -0.5, size = 3.2) +
  scale_fill_manual(values = setNames(unname(cols.region), unname(region_labels)), name = NULL) +
  scale_y_continuous(limits = c(0, 100), expand = expansion(mult = c(0.02, 0.12))) +
  labs(title = sprintf("Network perturbation percentile of target DEGs after %s virtual knockout", REP_TF),
       subtitle = "Percentile ranked by FC among all tested genes within each KO region (relative rank, not perturbation magnitude)",
       x = NULL, y = "Perturbation percentile (%)") +
  theme_bw(base_size = BASE_SIZE) +
  theme(legend.position = "top")
print(p4)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot4_target_gene_perturbation_percentile.pdf")),
       p4, width = 4.6, height = 3.8)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot4_target_gene_perturbation_percentile.png")),
       p4, width = 4.6, height = 3.8, dpi = 300)

## 指定 TF–target 配对的扰动可视化（NFIC→DIO2 / RORB→FNBP1 / ZIC1→MT2A）

展示指定 target 基因在对应 TF virtual KO 网络中的相对扰动排名（perturbation percentile：各 KO 全部 tested genes 为背景、按 FC 降序换算，top = 100%）。

- NFIC → DIO2：MG GRN（central / peripheral）
- RORB → FNBP1：MG GRN（central / peripheral）
- ZIC1 → MT2A：Cone GRN（仅 peripheral；central 的 KO 因 ZIC1 被 scQC 过滤未运行）
- EGR1 → PDE10A：完全跳过——EGR1(BC) KO 因 EGR1 检出率低于 minPCT=0.05 被 scQC 过滤未运行，原因详见 `2026-10-14_multiTF_KO_skip_reasons.md`

注意：百分位只反映各自网络内的相对排名（network perturbation），不代表扰动绝对强度或表达上下调；MG 与 Cone 为不同细胞类型的独立 GRN，百分位数值不可跨细胞类型直接比较。本模块只在 Notebook 内打印参数表，不生成新的数据表文件；图片输出 PDF + PNG。

In [ ]:
# ---- 12. 指定 TF-target 配对的扰动百分位与完整参数 ------------------------------
# EGR1-PDE10A 因 EGR1(BC) KO 被 scQC 过滤未运行，本模块完全跳过（原因见 2026-10-14_multiTF_KO_skip_reasons.md）
TARGET_PAIRS <- data.table(
  TF          = c("NFIC", "NFIC", "RORB", "RORB", "ZIC1"),
  target_gene = c("DIO2", "DIO2", "FNBP1", "FNBP1", "MT2A"),
  celltype    = c("MG", "MG", "MG", "MG", "Cone"),
  region      = c("central", "peripheral", "central", "peripheral", "peripheral")
)

## 读取各配对对应 KO 的完整 diffRegulation（文件发现不限日期前缀；缺失或基因不在检验集中 -> warning + NA）
pair_res <- rbindlist(lapply(seq_len(nrow(TARGET_PAIRS)), function(i) {
  tp <- TARGET_PAIRS[i]
  f <- list.files(KNK_DIR, pattern = paste0("_", tp$celltype, "_", tp$region, "_",
                                            tp$TF, "_diffRegulation\\.tsv$"), full.names = TRUE)
  if (length(f) > 1) {
    warning(sprintf("%s (%s, %s) 匹配到多个文件，使用最新: %s",
                    tp$TF, tp$celltype, tp$region, basename(sort(f)[length(f)])))
    f <- sort(f)[length(f)]
  }
  d <- NULL
  if (length(f) == 1) {
    d <- fread(f)
    miss <- setdiff(c("gene", "distance", "FC", "p.adj"), names(d))
    if (length(miss) > 0) stop(basename(f), " 缺少必需列: ", paste(miss, collapse = ", "))
  } else {
    warning(sprintf("%s (%s, %s) 的 diffRegulation 文件不存在，该行记为 NA",
                    tp$TF, tp$celltype, tp$region))
  }
  ok <- !is.null(d) && tp$target_gene %in% d$gene
  if (!is.null(d) && !ok) {
    warning(sprintf("%s 不在 %s (%s, %s) 的检验基因中，该行记为 NA",
                    tp$target_gene, tp$TF, tp$celltype, tp$region))
  }
  if (!ok) {
    out <- data.table(distance = NA_real_, FC = NA_real_, p.adj = NA_real_,
                      Perturbation_percentile = NA_real_, Significant = NA)
  } else {
    dd <- d[gene != tp$TF]   # 排除被敲除的 TF 自身
    rk <- frank(-dd$FC, ties.method = "first")
    pct <- round(100 * (length(rk) - rk) / pmax(length(rk) - 1, 1), 1)
    j <- which(dd$gene == tp$target_gene)
    out <- data.table(distance = dd$distance[j], FC = dd$FC[j], p.adj = dd$p.adj[j],
                      Perturbation_percentile = pct[j], Significant = dd$p.adj[j] < PADJ_CUTOFF)
  }
  cbind(tp, out)
}))

pair_res[, pair_label := sprintf("%s -> %s (%s)", TF, target_gene, celltype)]
pair_res[, Region := factor(region_labels[region], levels = unname(region_labels))]
pair_res[, region := NULL]
pair_res[, pair_label := factor(pair_label, levels = unique(pair_label))]
setcolorder(pair_res, c("pair_label", "TF", "target_gene", "celltype", "Region", "distance", "FC",
                        "p.adj", "Perturbation_percentile", "Significant"))
print(pair_res)

In [ ]:
# ---- 13. Plot 5: 指定 TF-target 配对的扰动百分位柱状图 --------------------------
p5 <- ggplot(pair_res[!is.na(Perturbation_percentile)],
             aes(pair_label, Perturbation_percentile, fill = Region)) +
  geom_col(position = position_dodge(0.75), width = 0.65, color = "grey20", linewidth = 0.2) +
  geom_text(aes(label = sprintf("%.1f", Perturbation_percentile)),
            position = position_dodge(0.75), vjust = -0.5, size = 3.2) +
  scale_fill_manual(values = setNames(unname(cols.region), unname(region_labels)), name = NULL) +
  scale_y_continuous(limits = c(0, 100), expand = expansion(mult = c(0.02, 0.12))) +
  labs(title = "Network perturbation percentile of target genes after virtual knockout",
       subtitle = "Percentile ranked by FC among all tested genes within each KO network; values not directly comparable across cell types",
       x = NULL, y = "Perturbation percentile (%)") +
  theme_bw(base_size = BASE_SIZE) +
  theme(legend.position = "top",
        axis.text.x = element_text(size = 8.5))
print(p5)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot5_target_pair_perturbation_percentile.pdf")),
       p5, width = 5.2, height = 3.8)
ggsave(file.path(PLOT_DIR, paste0(DATE, "_Plot5_target_pair_perturbation_percentile.png")),
       p5, width = 5.2, height = 3.8, dpi = 300)